# Bài 4 v4 · Gradient Descent
PGS.TS. Nguyễn Văn Hậu. Chạy lần lượt 10 ô mã. Dữ liệu thật và ví dụ mô phỏng được phân biệt; giữ số liệu và tọa độ của slide.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"source/gd_lab.py").exists():
    raise RuntimeError("Mở notebook tại thư mục Bài 4 đã giải nén")
if (ROOT/".build/pydeps").exists(): sys.path.insert(0,str(ROOT/".build/pydeps"))
sys.path.insert(0,str(ROOT/"source"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from gd_lab import toy_one_parameter, batch_gd, metrics, run
plt.rcParams.update({"figure.figsize":(9,5),"axes.grid":True})
OUT=ROOT/"results/lab_figures"; OUT.mkdir(parents=True,exist_ok=True)

Matplotlib is building the font cache; this may take a moment.


## 1. Tính tay và kiểm gradient
Toy x theo 100 km; y ngày; a=5, z=x−2,5. Gradient phải được tính lại tại mỗi trạng thái.

In [2]:
h=toy_one_parameter(.4,8)
print(pd.DataFrame([{k:r[k] for k in ["t","w","g","J"]} for r in h[:3]]).to_string(index=False))
assert np.isclose(h[1]["w"],.7) and np.isclose(h[2]["J"],.1015625)

 t    w       g        J
 0 0.00 -1.7500 1.250000
 1 0.70 -0.8750 0.331250
 2 1.05 -0.4375 0.101562


In [3]:
x=np.arange(1.,5.); y=np.array([3.,4.,6.,7.]); z=x-2.5
fig,ax=plt.subplots();ax.scatter(x,y,label="Mo phong");
for r in h[:3]: ax.plot(x,r["predictions"],label=f"t={r[chr(116)]}; w={r[chr(119)]:.2f}")
ax.set(xlabel="x / 100 km",ylabel="y, prediction / day");ax.legend();fig.tight_layout();fig.savefig(OUT/"toy_predictions.png",dpi=240);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_9764\2255211530.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="x / 100 km",ylabel="y, prediction / day");ax.legend();fig.tight_layout();fig.savefig(OUT/"toy_predictions.png",dpi=240);plt.show()


In [4]:
J=lambda w: np.mean((5+w*z-y)**2)/2
g=lambda w: np.mean(z*(5+w*z-y))
for w in [0,.7,1.4,2.1]:
    fd=(J(w+1e-5)-J(w-1e-5))/2e-5
    assert np.isclose(fd,g(w),atol=1e-8)
    print(w,g(w),fd)

0 -1.75 -1.7500000000114644
0.7 -0.8750000000000002 -0.8750000000168344
1.4 -2.220446049250313e-16 -3.816391647148976e-12
2.1 0.8750000000000002 0.8749999999890788


## 2. Learning rate
Cùng dữ liệu và khởi tạo; dùng chung thang để so. Overshoot không đồng nghĩa phân kỳ.

In [5]:
fig,axes=plt.subplots(2,1,figsize=(10,8));
for eta in [.1,.4,1.2,1.8]:
    hs=toy_one_parameter(eta,8); t=[r["t"] for r in hs]
    axes[0].plot(t,[r["w"] for r in hs],marker="o",label=f"eta={eta}")
    axes[1].semilogy(t,[r["J"] for r in hs],marker="o",label=f"eta={eta}")
axes[0].axhline(1.4,color="gray",ls="--");axes[0].set_ylabel("w / day per 100 km");axes[1].set(xlabel="step t",ylabel="J / day squared");axes[0].legend();fig.tight_layout();fig.savefig(OUT/"learning_rates.png",dpi=240);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_9764\3885728915.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[0].axhline(1.4,color="gray",ls="--");axes[0].set_ylabel("w / day per 100 km");axes[1].set(xlabel="step t",ylabel="J / day squared");axes[0].legend();fig.tight_layout();fig.savefig(OUT/"learning_rates.png",dpi=240);plt.show()


## 3. Hai tham số từ cùng trạng thái cũ

In [6]:
w,b,h2=batch_gd(x,y,eta=.05,steps=1,w0=.5,b0=1)
print(pd.DataFrame(h2).to_string(index=False))
assert np.allclose([w,b],[.9,1.1375])
H=np.array([[7.5,2.5],[2.5,1.]])
print("Hessian eigenvalues",np.linalg.eigvalsh(H))

 t   w      b      g_w     g_b        J
 0 0.5 1.0000 -8.00000 -2.7500 4.312500
 1 0.9 1.1375 -4.65625 -1.6125 1.481328
Hessian eigenvalues [0.14969513 8.35030487]


## 4. Olist thật
Để dựng CSV từ ZIP gốc, chạy `python source/prepare_olist.py` trước. Ô sau chạy đầy đủ lab từ CSV, kiểm dữ liệu, gradient, split và chỉ số.

In [7]:
audit=run(rebuild=False)
print(audit["status"],len(audit["checks"]))
print(pd.DataFrame(audit["real"]["four_orders"]).to_string(index=False))

pass 39
                        order_id  distance_km  delivery_days  promised_days  prediction_GD       order_approved_at order_delivered_customer_date
fc9bc4bac939adaa600ad59dee9a106f   100.028002       1.814294       7.197801       8.472040 2017-05-16T19:15:10.000       2017-05-18T14:47:45.000
67f2f913bb16abed933504fb19d4ade4   399.992287       6.911921      21.159572      10.163538 2017-07-19T20:10:13.000       2017-07-26T18:03:23.000
7d09a3b11e55c5e586d952edc9e12e6b   800.017914      14.719606      25.367836      12.419282 2017-05-21T15:10:19.000       2017-06-05T08:26:33.000
34445bc5a6255519c1d230b2231eeb63  1400.105510      11.548611      49.984167      15.803175 2017-02-05T00:22:48.000       2017-02-16T13:32:48.000


In [8]:
train=pd.read_csv(ROOT/"data/prepared/olist_model_table.csv");train=train.loc[train["split"]=="train"].copy()
fig,ax=plt.subplots(); valid=train.dropna(subset=["distance_km","delivery_days"])
ax.hexbin(valid.distance_km,valid.delivery_days,gridsize=(30,18),extent=(0,3000,0,45),mincnt=1,bins="log",cmap="Blues")
r=audit["real"];xx=np.linspace(0,3000,200);ax.plot(xx,r["b_raw"]+r["w_raw"]*xx,color="#bc761a",label="GD 100")
ax.set(xlim=(0,3000),ylim=(0,45),xlabel="distance / km",ylabel="delivery from approval / day");ax.legend();fig.tight_layout();fig.savefig(OUT/"olist_density.png",dpi=240);plt.show()

C:\Users\nguye\AppData\Local\Temp\ipykernel_9764\876518594.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlim=(0,3000),ylim=(0,45),xlabel="distance / km",ylabel="delivery from approval / day");ax.legend();fig.tight_layout();fig.savefig(OUT/"olist_density.png",dpi=240);plt.show()


In [9]:
history=pd.DataFrame(audit["real"]["history"])
fig,ax=plt.subplots();ax.plot(history.t,history.J,label="Batch GD eta=.1");ax.set(xlabel="step t",ylabel="train J / day squared");ax.legend();fig.tight_layout();fig.savefig(OUT/"olist_GD_loss.png",dpi=240);plt.show()
print(audit["real"]["metrics"])

{'validation': {'GD': {'MAE_days': 4.683002441494434, 'RMSE_days': 6.37566511729293}, 'baseline': {'MAE_days': 5.364407287174175, 'RMSE_days': 7.033250138582075}, 'OLS': {'MAE_days': 4.682999025444396, 'RMSE_days': 6.375606535473678}}, 'test': {'GD': {'MAE_days': 5.9396250620880835, 'RMSE_days': 9.351597901427297}, 'baseline': {'MAE_days': 6.851263997567359, 'RMSE_days': 10.16748570384581}, 'OLS': {'MAE_days': 5.939644970338745, 'RMSE_days': 9.351535869379271}}}


C:\Users\nguye\AppData\Local\Temp\ipykernel_9764\3470473130.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.plot(history.t,history.J,label="Batch GD eta=.1");ax.set(xlabel="step t",ylabel="train J / day squared");ax.legend();fig.tight_layout();fig.savefig(OUT/"olist_GD_loss.png",dpi=240);plt.show()


## 5. Đọc lỗi và kết luận
MAE là trung bình; RMSE nhạy với sai số lớn. Test lịch sử đã được xem từ phiên bản trước; đây là tái lập. Dữ liệu là nhóm đã giao đủ điều kiện, không đại diện tự động cho mọi đơn triển khai.

In [10]:
pred=pd.read_csv(ROOT/"data/test_predictions.csv")
print(pred.columns.tolist())
e=pred["prediction_GD"]-pred["delivery_days"]
fig,ax=plt.subplots();ax.hist(np.clip(e,-40,40),bins=np.linspace(-40,40,21),color="#245a84");ax.set(xlabel="GD residual / day (outside +/-40 merged at edges)",ylabel="test orders");fig.tight_layout();fig.savefig(OUT/"test_residuals.png",dpi=240);plt.show()
print("MAE",np.abs(e).mean(),"RMSE",np.sqrt(np.mean(e**2)))
assert len(e)==40266
assert np.isclose(np.abs(e).mean(),5.9396250620880835)

['order_id', 'split', 'distance_km', 'delivery_days', 'prediction_GD', 'prediction_baseline', 'residual_GD']


MAE 5.9396250620880835 RMSE 9.351597901427297


C:\Users\nguye\AppData\Local\Temp\ipykernel_9764\3482182055.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.hist(np.clip(e,-40,40),bins=np.linspace(-40,40,21),color="#245a84");ax.set(xlabel="GD residual / day (outside +/-40 merged at edges)",ylabel="test orders");fig.tight_layout();fig.savefig(OUT/"test_residuals.png",dpi=240);plt.show()
